# PerturbAtlas Benchmark Summary

This notebook provides a release-level benchmark summary for **PerturbAtlas-K562-v1.0**.

It includes:

1. Loading the released atlas.
2. Reporting the five validation benchmarks.
3. Running small reproducibility/sanity checks on the released atlas object.
4. Saving a benchmark summary table.

The full validation experiments were developed in Notebook 12. This notebook is intentionally lightweight and suitable for GitHub users.


## Setup notes

If you see `ModuleNotFoundError: No module named 'perturbatlas'`, install the package from the repository root with:

```python
%pip install -e .
```

If you run the notebook from inside the `examples/` folder, use:

```python
%pip install -e ..
```

If you see `ModuleNotFoundError: No module named 'pandas'`, install dependencies with:

```python
%pip install -r requirements.txt
```

or from inside `examples/`:

```python
%pip install -r ../requirements.txt
```


## 1. Import and load the atlas

In [ ]:
import pandas as pd
from perturbatlas import K562Atlas

In [ ]:
from pathlib import Path

def find_repo_root():
    """Find repository root from either the project root or examples/ folder."""
    cwd = Path.cwd().resolve()

    if (cwd / "data" / "PerturbAtlas-K562-v1.0.pkl").exists():
        return cwd

    if (cwd.parent / "data" / "PerturbAtlas-K562-v1.0.pkl").exists():
        return cwd.parent

    raise FileNotFoundError(
        "Could not find data/PerturbAtlas-K562-v1.0.pkl. "
        "Please run this notebook from the repository root or examples/ folder."
    )

ROOT = find_repo_root()
ATLAS_PATH = ROOT / "data" / "PerturbAtlas-K562-v1.0.pkl"

print("Repository root:", ROOT)
print("Atlas path:", ATLAS_PATH)


In [ ]:
atlas = K562Atlas.load(ATLAS_PATH)

atlas.info()

## 2. Validation overview

The following validation experiments were completed before the v1.0 release:

1. Leave-One-Perturbation-Out validation
2. Random gene-set controls
3. Query-size robustness
4. Noise robustness
5. Cross-module specificity


In [ ]:
benchmark_summary = pd.DataFrame([
    {
        "benchmark": "Leave-One-Perturbation-Out",
        "purpose": "Recover known perturbation programs",
        "main_result": "90.7% module accuracy; 98.1% top-3/top-5 retrieval"
    },
    {
        "benchmark": "Random gene controls",
        "purpose": "Reject meaningless gene sets",
        "main_result": "Mean evidence ≈ 0.06; no high-confidence random calls"
    },
    {
        "benchmark": "Query-size robustness",
        "purpose": "Measure performance across query sizes",
        "main_result": "≥20 genes reached 100% module accuracy"
    },
    {
        "benchmark": "Noise robustness",
        "purpose": "Test robustness to irrelevant genes",
        "main_result": "20 signal genes + 80 random genes retained 96.3% accuracy"
    },
    {
        "benchmark": "Cross-module specificity",
        "purpose": "Evaluate competing biological programs",
        "main_result": "Evidence decreased near balanced mixtures and recovered with dominant programs"
    }
])

benchmark_summary

## 3. LOPO benchmark summary

In [ ]:
lopo_summary = pd.DataFrame([
    {"metric": "Module prediction accuracy", "value": "90.7%"},
    {"metric": "Top-1 module retrieval", "value": "79.6%"},
    {"metric": "Top-3 module retrieval", "value": "98.1%"},
    {"metric": "Top-5 module retrieval", "value": "98.1%"},
])

lopo_summary

## 4. Query-size robustness summary

In [ ]:
query_size_summary = pd.DataFrame([
    {"query_size": 5, "module_accuracy": "90.7%"},
    {"query_size": 10, "module_accuracy": "98.3%"},
    {"query_size": 20, "module_accuracy": "100%"},
    {"query_size": 40, "module_accuracy": "100%"},
    {"query_size": 80, "module_accuracy": "100%"},
])

query_size_summary

## 5. Sanity check: known perturbation-derived queries

In [ ]:
test_perturbations = ["CEBPA", "KLF1", "HOXA13"]

sanity_rows = []

for perturbation in test_perturbations:
    query_genes = list(atlas.strong_up[perturbation])[:40]
    result = atlas.query(query_genes)

    sanity_rows.append({
        "source_perturbation": perturbation,
        "predicted_module": result.predicted_module,
        "evidence_score": round(result.evidence_score(), 3),
        "n_retrieved": len(result.all_results),
        "top_perturbation": (
            result.all_results.iloc[0]["perturbation"]
            if len(result.all_results) > 0
            else None
        )
    })

sanity_check = pd.DataFrame(sanity_rows)
sanity_check

## 6. Sanity check: low-evidence query

In [ ]:
weak_query = ["BAK1", "CEBPA", "SPI1", "KLF1"]

weak_result = atlas.query(weak_query)

weak_result.summary()
weak_result.interpret()

## 7. Export benchmark summary

In [ ]:
output_path = ROOT / "examples" / "benchmark_summary.csv"

benchmark_summary.to_csv(output_path, index=False)

print("Saved:", output_path)

## Notes

This notebook is intended as a compact release benchmark summary.

The full validation experiments from Notebook 12 include:

- complete LOPO validation,
- random gene-set controls,
- query-size robustness,
- noise robustness,
- cross-module specificity.

